Structured output



Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

Pydantic


Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:openai/gpt-oss-120b")


In [2]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movies rating out of 10")

In [9]:
model_with_structure=model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.6', 'langchain': '1.3.15'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001A2C89A3230>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001A2C89A3CB0>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The t

In [10]:
res = model_with_structure.invoke("provide info about the movie 3 idiots")
res

Movie(title='3 Idiots', year=2009, director='Rajkumar Hirani', rating=8.5)

without parsed ouput with parsed output 
** include_raw=True**

In [12]:
model_with_structure_both=model.with_structured_output(Movie, include_raw=True)
model_with_structure_both

{
  raw: _ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.6', 'langchain': '1.3.15'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001A2C89A3230>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001A2C89A3CB0>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description

In [15]:
res_both = model_with_structure_both.invoke("info about the dangal movie")
res_both

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'User wants info about the "Dangal" movie. We should provide details: director, rating, title, year. Possibly also summary. We can use the provided function Movie to return structured info. Use function call.', 'tool_calls': [{'id': 'fc_21b77a18-4330-42fe-af91-6d62ab04a0fd', 'function': {'arguments': '{"director":"Nitesh Tiwari","rating":8.5,"title":"Dangal","year":2016}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 101, 'prompt_tokens': 157, 'total_tokens': 258, 'completion_time': 0.214135894, 'completion_tokens_details': {'reasoning_tokens': 45}, 'prompt_time': 0.006381267, 'prompt_tokens_details': None, 'queue_time': 0.368847286, 'total_time': 0.220517161}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_8655ddce88', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a02d84-003d-7

NESTED STRUCTURE

In [16]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: list[str]
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movies which are highest grossing in 2000")

In [17]:
response

MovieDetails(title=['Mission: Impossible 2'], year=2000, cast=[Actor(name='Tom Cruise', role='Ethan Hunt'), Actor(name='Thandie Newton', role='Nyah'), Actor(name='Dougray Scott', role='Sean Ambrose')], genres=['Action', 'Adventure', 'Thriller'], budget=125000000.0)